# Démo xyt_gps avec test set générique

Ce notebook montre le flux complet du package sur un jeu GPS compact : 15 utilisateurs, une expérimentation `test_set`, trois phases au printemps 2026.

Le test set est généré localement depuis `Data/Output/0-landed-data/<experiment_name>/anonymized_altered`. Il est repseudonymisé, recalé temporellement et déplacé spatialement une seconde fois. Les traces du profil `anonymized_altered` sont déjà altérées et ne doivent pas être interprétées scientifiquement. Les fichiers produits restent dans `Data/Output/test-set-gps`, ignoré par git.


## 1. Préparer l'environnement

La cellule retrouve automatiquement le package local, que le notebook soit lancé depuis la racine du dépôt ou depuis `package-xyt-gps/examples`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd


def find_package_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for base in (start, *start.parents):
        for candidate in (base, base / "package-xyt-gps"):
            if (candidate / "src" / "xyt_gps").exists():
                return candidate.resolve()
    raise FileNotFoundError("Impossible de trouver package-xyt-gps/src/xyt_gps.")


package_root = find_package_root()
import xyt_gps as xyt
pd.set_option("display.max_columns", 80)
print(f"xyt_gps chargé depuis : {xyt.__file__}")
package_root

## 2. Générer ou retrouver le test set

Si `Data/Output/test-set-gps` existe déjà, le notebook l'utilise directement. Sinon il lance le générateur local.

In [ ]:
test_set_dir = (package_root.parent / "Data" / "Output" / "test-set-gps").resolve()
if not test_set_dir.exists():
    generator = package_root / "examples" / "build_generic_test_set.py"
    subprocess.run([sys.executable, str(generator)], check=True)

manifest = json.loads((test_set_dir / "test_set_manifest.json").read_text(encoding="utf-8"))
manifest["tables"]

## 3. Charger les tables GPS

Le test set reprend la structure attendue par le package : `storyline`, `trips`, `journeys`, `user_statistics`, `user_presence`.

In [ ]:
storyline = pd.read_parquet(test_set_dir / "storyline.parquet")
trips = pd.read_parquet(test_set_dir / "trips.parquet")
journeys = pd.read_parquet(test_set_dir / "journeys.parquet")
user_statistics = pd.read_csv(test_set_dir / "user_statistics.csv", low_memory=False)
user_presence = pd.read_parquet(test_set_dir / "user_presence.parquet")

summary = pd.DataFrame(
    [
        {"table": "storyline", "rows": len(storyline), "users": storyline["user_id"].nunique()},
        {"table": "trips", "rows": len(trips), "users": trips["user_id"].nunique()},
        {"table": "journeys", "rows": len(journeys), "users": journeys["user_id"].nunique()},
        {"table": "user_statistics", "rows": len(user_statistics), "users": user_statistics["user_id"].nunique()},
        {"table": "user_presence", "rows": len(user_presence), "users": user_presence["user_id"].nunique()},
    ]
)
summary

## 4. Valider le schéma d'entrée

`expected_gps_schema()` donne le contrat de colonnes. `check_raw_import_columns()` vérifie les tables avant transformation.

In [ ]:
xyt.expected_gps_schema().head(20)

In [ ]:
schema_report = xyt.check_raw_import_columns(
    storyline,
    user_statistics,
    trips=trips,
    journeys=journeys,
    raise_on_error=True,
)
schema_report

## 5. Définir la configuration projet

La configuration décrit ici une expérimentation générique avec trois phases. Sur un projet sans phase, on peut utiliser `xyt.ProjectConfig()` sans `phases`.

In [ ]:
landing_config = json.loads((test_set_dir / "landing_config.json").read_text(encoding="utf-8"))
experiment = landing_config["experiments"][0]

phases = []
for phase_index in (1, 2, 3):
    start, end = experiment[f"phase_{phase_index}"]
    phases.append(xyt.Phase(f"Phase{phase_index}", start, end))

config = xyt.ProjectConfig(
    experiment_name=experiment["experiment_name"],
    motiontag_project_name=experiment["motiontag_project_name"],
    period=experiment["dump_date_range"],
    raw_data_dir=test_set_dir,
    phases=tuple(phases),
    mappings=xyt.mode_purpose_mapping(),
)
config

## 6. Transformer les données GPS

`prepare_mobility_dataset()` orchestre les étapes principales tout en gardant les tables intermédiaires inspectables : storyline préparée, legs, staypoints, trips, journeys, mappings et user_stats.

In [ ]:
raw = xyt.RawGpsData(
    storyline=storyline,
    trips=trips,
    journeys=journeys,
    user_statistics=user_statistics,
)

dataset = xyt.prepare_mobility_dataset(
    raw,
    config,
    resample_missing_days=True,
    clean_leg_geometries=True,
    add_length_outlier_flags=True,
    add_signal_quality_flags=True,
)

{x: len(table) for x, table in xyt.mobility_dataset_tables(dataset).items()}

## 7. Contrôler la participation

La grille compte, pour chaque utilisateur et semaine de phase, le nombre de jours avec tracking observé. Les jours resamplés ne sont pas comptés comme jours observés.

In [ ]:
participation = xyt.build_weekly_participation_grid(
    dataset.storyline,
    config,
    user_ids=dataset.user_stats["user_id"],
    exclude_resampled=True,
)
xyt.summarize_participation_grid(participation)

In [ ]:
xyt.plot_participation_heatmap(
    participation,
    title="Participation hebdomadaire du test set",
)

## 8. Calculer les indicateurs

Les indicateurs sont produits à trois grains : personne-jour, personne-phase et population.

In [ ]:
indicators = xyt.compute_mobility_indicators(
    dataset,
    config=config,
    mode_col="mode_niv1",
    include_zero_days=True,
)

{
    "person_day": indicators.person_day.shape,
    "person_phase": indicators.person_phase.shape,
    "population": indicators.population.shape,
}

In [ ]:
xyt.population_indicator_summary(indicators).head(20)

In [ ]:
xyt.plot_indicator_bars(
    indicators,
    table="population",
    metrics=["distance_km_per_day"],
    group_col="mode",
    facet_col="phase",
    title="Distance moyenne journalière par phase et mode",
)

## 9. Exporter si nécessaire

Les exports restent optionnels dans une démo. Quand `EXPORT_DEMO = True`, les tables sont écrites dans `Data/Output/test-set-results`, un dossier ignoré par git.

In [ ]:
EXPORT_DEMO = False
result_dir = (package_root.parent / "Data" / "Output" / "test-set-results").resolve()

if EXPORT_DEMO:
    mobility_manifest = xyt.write_mobility_dataset(
        dataset,
        result_dir / "mobility",
        formats=("parquet", "csv"),
        extra_tables={"participation_grid": participation, "user_presence": user_presence},
    )
    indicator_manifest = xyt.write_indicator_result(
        indicators,
        result_dir / "indicators",
        formats=("parquet", "csv"),
    )
else:
    mobility_manifest = pd.DataFrame([{"message": "Mettre EXPORT_DEMO = True pour écrire les fichiers.", "target_dir": str(result_dir)}])
    indicator_manifest = pd.DataFrame()

mobility_manifest

## À retenir

Le package peut être utilisé par étapes : charger les tables, valider le schéma, définir une configuration, transformer les données, contrôler la participation, puis produire les indicateurs. Le test set sert à apprendre ce flux sans manipuler directement un export complet.